# Source Inspection

Quick inspection of the raw CSV to inform transformation decisions - which fields are multi-value, how dates are formatted, what needs splitting into child tables

**Source:** COVID-19 Clinical Trials (Kaggle, Parul Pandey) - 5,783 studies, 27 columns  
**Target:** the 7-table relational schema specified in the challenge brief

In [2]:
import pandas as pd 
import numpy as np 

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', None)

PATH = "../data/raw/COVID clinical trials.csv"

df = pd.read_csv(PATH)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5783 entries, 0 to 5782
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Rank                     5783 non-null   int64  
 1   NCT Number               5783 non-null   str    
 2   Title                    5783 non-null   str    
 3   Acronym                  2480 non-null   str    
 4   Status                   5783 non-null   str    
 5   Study Results            5783 non-null   str    
 6   Conditions               5783 non-null   str    
 7   Interventions            4897 non-null   str    
 8   Outcome Measures         5748 non-null   str    
 9   Sponsor/Collaborators    5783 non-null   str    
 10  Gender                   5773 non-null   str    
 11  Age                      5783 non-null   str    
 12  Phases                   3322 non-null   str    
 13  Enrollment               5749 non-null   float64
 14  Funded Bys               5783 non-n

### Observations

- All date fields are strings - non-standard formats, explicit parsing required.
- `Enrollment` is float64 due to nulls; target schema requires INTEGER.
- Target schema applies VARCHAR length limits across most string fields. 
- Postgres rejects oversized values rather than truncating - lengths need checking before load.

In [3]:
# Preview actual records
df.head(5).T

,0,1,2,3,4
Rank,1,2,3,4,5
NCT Number,NCT04785898,NCT04595136,NCT04395482,NCT04416061,NCT04395924
Title,Diagnostic Performance of the ID Now™ COVID-19 Screening Test Versus Simplexa™ COVID-19 Direct Assay,Study to Evaluate the Efficacy of COVID19-0001-USR in Patients With Mild/or Moderate COVID-19 Infection in Outpatient,Lung CT Scan Analysis of SARS-CoV2 Induced Lung Injury,The Role of a Private Hospital in Hong Kong Amid COVID-19 Pandemic,Maternal-foetal Transmission of SARS-Cov-2
Acronym,COVID-IDNow,COVID-19,TAC-COVID19,COVID-19,TMF-COVID-19
Status,"Active, not recruiting",Not yet recruiting,Recruiting,"Active, not recruiting",Recruiting
Study Results,No Results Available,No Results Available,No Results Available,No Results Available,No Results Available
Conditions,Covid19,SARS-CoV-2 Infection,covid19,COVID,Maternal Fetal Infection Transmission|COVID-19|SARS-CoV 2
Interventions,Diagnostic Test: ID Now™ COVID-19 Screening Test,Drug: Drug COVID19-0001-USR|Drug: normal saline,Other: Lung CT scan analysis in COVID-19 patients,Diagnostic Test: COVID 19 Diagnostic Test,"Diagnostic Test: Diagnosis of SARS-Cov2 by RT-PCR and : IgG, Ig M serologies in the amniotoc fluid, the blood cord and the placenta"
Outcome Measures,Evaluate the diagnostic performance of the ID Now ™ COVID-19 test carried out by nurses in an emergency department in comparison with the reference PCR test: Simplexa ™ COVID-19 Direct,Change on viral load results from baseline after using COVID19-0001-USR via nebulization,A qualitative analysis of parenchymal lung damage induced by COVID-19|A quantitative analysis of parenchymal lung damage induced by COVID-19|The potential impact of parenchymal morphological CT sc...,Proportion of asymptomatic subjects|Proportion of subjects with recent contact history|Proportion of subjects with recent travel history,COVID-19 by positive PCR in cord blood and / or positive serologies|COVID-19 by positive PCR in amniotic fluid and placenta|New born infected by COVID-19
Sponsor/Collaborators,Groupe Hospitalier Paris Saint Joseph,United Medical Specialties,University of Milano Bicocca,Hong Kong Sanatorium & Hospital,Centre Hospitalier Régional d'Orléans|Centre de Biophysique Moléculaire - Pr Chantal Pichon|Professeur TOUMI Hechmi


### Fields needing transformation

- Most multi-value fields are pipe-delimited and split into child tables: `Conditions`, `Outcome Measures`, `Sponsor/Collaborators` (first entry is lead), and `Interventions` (formatted `Type: Name`).
- `Locations` has pipe-separated sites, each with comma-separated parts, but the number of parts varies.
- `Study Designs` holds `Key: Value` pairs mapping to the five `study_design` columns. Interventional and observational studies use different keys.
- `Age` gives a range plus a bucket ("18 Years and older (Adult, Older Adult)"); the bucket has no schema field and will be dropped.
- Dates are mostly full ("November 9, 2020") but some are month-only.

In [4]:
# Check Locations column: count how many comma-separated parts each individual location has.

loc_parts = df['Locations'].dropna().str.split('|').explode().str.count(',') + 1

print(loc_parts.value_counts().sort_index())

Locations
3      9255
4     13153
5      1403
6       247
7        36
8        11
9         6
10        4
Name: count, dtype: int64


In [5]:
# Locations vary from 3 to 10 comma-separated parts. Sampling the higher counts
# to determine whether the extra parts are a structural difference or noise inside the facility name.

sites = df['Locations'].dropna().str.split('|').explode()

for n in [5, 6, 7, 8, 9, 10]:
    print(f"===== {n} parts =====")
    for val in sites[sites.str.count(',') + 1 == n].head(3):
        print(repr(val))
    print()

===== 5 parts =====
'Prince of Wales Hospital, The Chinese University of Hong Kong, Sha Tin, New Territories, Hong Kong'
'Regional Hospital West Jutland, Hostebro, Holstebro, DK, Denmark'
'Regional Hospital Central Jutland, Viborg, Viborg, DK, Denmark'

===== 6 parts =====
'Wits RHI, University of the Witwatersrand, Hillbrow, Johannesburg,Gauteng, South Africa'
'Mount Sinai Hospital, Department of Medicine, Division of Clinical Immunology, New York, New York, United States'
"Boston Children's Hospital: School Inner-City Asthma Study (SICAS-2), Environmental Assessment of Sleep in Youth (EASY), Severe Asthma Research Program (SARP) and Preventing Asthma in High Risk Kids (PARK) Site, Boston, Massachusetts, United States"

===== 7 parts =====
'Kings County Hospital Center, Brooklyn, NY, USA, Albertson, New York, United States'
'Virginia Commonwealth University, Department of Internal Medicine, Division of Rheumatology, Allergy & Immunology, Richmond, Virginia, United States'
'Department 

### Location parsing

Sites range from 3 to 10 comma-separated parts. The extras come from commas inside facility names - department lists, institutional hierarchies - not from additional structured fields.

Two defects surfaced:

- **Duplicated address data.** Some facility values contain the full address, which then repeats in the structured fields: 
  `Cardiology Unit, ... , Milan, Italy, Milan, Lombardia, Italy`. 
- **Malformed delimiters.** `Johannesburg,Gauteng` - missing space after comma.

**Approach:** parse right-anchored, since the trailing fields are reliable. Last part is country; for 4+ parts the next two are state and city, for 3 parts there is no state. Everything remaining goes into `facility` unchanged.

Facility values are loaded as-is rather than cleaned, so the duplication remains detectable during profiling.

In [6]:
# Age packs a range and a category into one string. Checking what shapes exist before writing the parser.

print(df["Age"].value_counts().head(20))
print(df["Age"].value_counts().tail(20))
print(f"\nDistinct values: {df['Age'].nunique()}")

Age
18 Years and older   (Adult, Older Adult)           2885
Child, Adult, Older Adult                            486
18 Years to 80 Years   (Adult, Older Adult)          221
18 Years to 65 Years   (Adult, Older Adult)          155
18 Years to 75 Years   (Adult, Older Adult)          135
18 Years to 100 Years   (Adult, Older Adult)         119
18 Years to 70 Years   (Adult, Older Adult)          107
18 Years to 99 Years   (Adult, Older Adult)           92
18 Years to 90 Years   (Adult, Older Adult)           88
18 Years to 60 Years   (Adult)                        87
18 Years to 85 Years   (Adult, Older Adult)           80
16 Years and older   (Child, Adult, Older Adult)      61
19 Years and older   (Adult, Older Adult)             40
65 Years and older   (Older Adult)                    39
18 Years to 55 Years   (Adult)                        37
18 Years to 45 Years   (Adult)                        35
60 Years and older   (Adult, Older Adult)             35
50 Years and older   (Adult

### Age

417 distinct values across four structural shapes:

- `"18 Years and older (Adult, Older Adult)"` - minimum only
- `"18 Years to 80 Years (Adult, Older Adult)"` - both bounds
- `"up to 1 Year (Child)"` - maximum only
- `"Child, Adult, Older Adult"` - category only, no numeric range (486 records, 8.4%)

Units are mixed - Years, Months, and occasionally both within one record (`"3 Months to 18 Years"`). Singular and plural both appear.

**Approach:** store the range strings as they are, units intact, matching the schema's VARCHAR(20). The parenthetical category is dropped - no schema field. Values would need normalising before any numeric comparison; noted as a limitation rather than resolved here.